# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [19]:
import pandas as pd

df = pd.read_csv("content_refresh_anonymized.csv")
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [20]:
ctr_table = (
    df.groupby(pd.qcut(df["ctr"], 4, duplicates="drop"))
      .agg(
          avg_position=("avg_position", "mean"),
          n=("ctr", "count")
      )
)

ctr_table

/tmp/ipykernel_6118/3408745717.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(pd.qcut(df["ctr"], 4, duplicates="drop"))


,avg_position,n
ctr,,
"(-0.001, 0.07]",19.218431,15224
"(0.07, 0.29]",15.116007,7503
"(0.29, 100.0]",11.587323,7273


Signal Check 1: CTR

Verdict: CONFIRMED

Observed: Lower CTR buckets tend to have worse average positions.

In [21]:
position_table = (
    df.groupby(pd.qcut(df["avg_position"], 4, duplicates="drop"))
      .agg(
          avg_ctr=("ctr", "mean"),
          n=("avg_position", "count")
      )
)

position_table

/tmp/ipykernel_6118/1789876622.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(pd.qcut(df["avg_position"], 4, duplicates="drop"))


,avg_ctr,n
avg_position,,
"(-0.001, 6.2]",1.079626,7543
"(6.2, 10.8]",0.436351,7534
"(10.8, 22.3]",0.319024,7462
"(22.3, 245.0]",0.202433,7461


Signal Check 2: Average Position

Verdict: CONFIRMED

Observed: Pages with poorer positions generally show lower CTR.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*




Rule:

I prioritize pages with low CTR and poor average position.

Observed pattern:
Pages with low CTR and positions beyond page one may benefit from optimization.

Reason Codes:

CTR_LOW = CTR below 0.20

POSITION_POOR = Average position above 20

CTR_AND_POSITION = Both conditions present

Action Label:

OPTIMIZE_CONTENT

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [22]:
import pandas as pd

df = pd.read_csv("content_refresh_anonymized.csv")

In [23]:
df["ctr"] = pd.to_numeric(df["ctr"], errors="coerce")
df["avg_position"] = pd.to_numeric(df["avg_position"], errors="coerce")

In [24]:
df["baseline_score"] = (
    (1 - df["ctr"]) * 70
    +
    (df["avg_position"] / df["avg_position"].max()) * 30
)

In [25]:
def reason_code(row):

    low_ctr = row["ctr"] < 0.20
    poor_pos = row["avg_position"] > 20

    if low_ctr and poor_pos:
        return "CTR_AND_POSITION"

    if low_ctr:
        return "CTR_LOW"

    if poor_pos:
        return "POSITION_POOR"

    return "REVIEW"

In [26]:
df["reason_code"] = df.apply(reason_code, axis=1)

df["action_label"] = "OPTIMIZE_CONTENT"

In [27]:
queue = df.sort_values(
    "baseline_score",
    ascending=False
)

In [28]:
import os

os.makedirs("work/outputs", exist_ok=True)

In [29]:
queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

In [30]:
queue.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,baseline_score,reason_code,action_label
24445,content_661e1745db72,client_e29c9c180c,2400.0,0.00,LOW,0.12,keyword article,commercial,1385.0,8651.0,...,0.0,50.0,0.0,low,deep,new,NaN,100.000000,CTR_AND_POSITION,OPTIMIZE_CONTENT
19920,content_23f1cc8851a9,client_e29c9c180c,30.0,0.25,LOW,0.00,keyword article,commercial,1626.0,10526.0,...,0.0,0.0,100.0,low,deep,new,NaN,92.530612,CTR_AND_POSITION,OPTIMIZE_CONTENT
26873,content_7275a6a3a8eb,client_e29c9c180c,10.0,0.00,LOW,0.00,keyword article,commercial,5737.0,37932.0,...,0.0,0.0,0.0,low,deep,new,NaN,90.265306,CTR_AND_POSITION,OPTIMIZE_CONTENT
16044,content_71a31b831092,client_e29c9c180c,30.0,0.01,LOW,0.00,keyword article,informational,1524.0,10012.0,...,0.0,0.0,0.0,low,deep,new,NaN,89.714286,CTR_AND_POSITION,OPTIMIZE_CONTENT
18532,content_42c7c72b8391,client_e29c9c180c,40.0,0.01,LOW,0.00,keyword article,transactional,1426.0,9353.0,...,0.0,0.0,0.0,low,deep,new,NaN,87.816327,CTR_AND_POSITION,OPTIMIZE_CONTENT


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

.1. Action: OPTIMIZE_CONTENT
Reason Code: CTR_AND_POSITION
Confidence: Medium
What would make it wrong: The page may target a niche keyword with naturally low CTR.

2. Action: OPTIMIZE_CONTENT
Reason Code: CTR_AND_POSITION
Confidence: Medium
What would make it wrong: Search demand may be too limited for optimization to have impact.

3. Action: OPTIMIZE_CONTENT
Reason Code: CTR_AND_POSITION
Confidence: Medium
What would make it wrong: Ranking may be constrained by strong competitors rather than content quality.

4. Action: OPTIMIZE_CONTENT
Reason Code: CTR_AND_POSITION
Confidence: Medium
What would make it wrong: The page may already satisfy user intent despite low CTR.

5. Action: OPTIMIZE_CONTENT
Reason Code: CTR_AND_POSITION
Confidence: Medium
What would make it wrong: CTR may be unstable due to low impressions.

Action: OPTIMIZE_CONTENT
Reason Code: CTR_AND_POSITION
Confidence: Medium
What would make it wrong: The page may have seasonal traffic patterns affecting CTR.
Action: OPTIMIZE_CONTENT
Reason Code: CTR_AND_POSITION
Confidence: Medium
What would make it wrong: Search intent may not match the current content type.
Action: OPTIMIZE_CONTENT
Reason Code: CTR_AND_POSITION
Confidence: Medium
What would make it wrong: The page may require technical SEO improvements rather than content changes.
Action: OPTIMIZE_CONTENT
Reason Code: CTR_AND_POSITION
Confidence: Medium
What would make it wrong: Low CTR may be caused by an unattractive title rather than content quality.
Action: OPTIMIZE_CONTENT
Reason Code: CTR_AND_POSITION
Confidence: Medium
What would make it wrong: The keyword may have limited click potential.
Action: OPTIMIZE_CONTENT
Reason Code: CTR_AND_POSITION
Confidence: Medium
What would make it wrong: Competitors may dominate the search results with stronger authority.
Action: OPTIMIZE_CONTENT
Reason Code: CTR_AND_POSITION
Confidence: Medium
What would make it wrong: Recent ranking fluctuations may not represent a long-term pattern.
Action: OPTIMIZE_CONTENT
Reason Code: CTR_AND_POSITION
Confidence: Medium
What would make it wrong: The page may already be improving without intervention.
Action: OPTIMIZE_CONTENT
Reason Code: CTR_AND_POSITION
Confidence: Medium
What would make it wrong: Low impressions may make CTR unreliable.
Action: OPTIMIZE_CONTENT
Reason Code: CTR_AND_POSITION
Confidence: Medium
What would make it wrong: The page could target a highly specific audience.
Action: OPTIMIZE_CONTENT
Reason Code: CTR_AND_POSITION
Confidence: Medium
What would make it wrong: External factors may influence ranking performance.
Action: OPTIMIZE_CONTENT
Reason Code: CTR_AND_POSITION
Confidence: Medium
What would make it wrong: The content may already satisfy user intent effectively.
Action: OPTIMIZE_CONTENT
Reason Code: CTR_AND_POSITION
Confidence: Medium
What would make it wrong: The page may need backlink improvements more than content updates.
Action: OPTIMIZE_CONTENT
Reason Code: CTR_AND_POSITION
Confidence: Medium
What would make it wrong: Search demand may have declined over time.
Action: OPTIMIZE_CONTENT
Reason Code: CTR_AND_POSITION
Confidence: Medium
What would make it wrong: The observed metrics may not capture the full business value of the page.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak Picks

Some pages received high scores because of very low CTR, but low traffic volume may reduce confidence in the recommendation.

Some pages may rank poorly because of competitive search results rather than content issues.

Leakage Check

No future information was used.

No outcome labels were used.

The score only uses observed CTR and average position values available at decision time.

This baseline is intended as a decision-support ranking rather than a prediction model.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.